# 07 · Proposed model: TFT + EWC (E9)

**Goal:** evaluate the proposed Temporal Fusion Transformer with Elastic Weight Consolidation under exactly the continual-learning protocol of notebook 06, together with the ablations that make its result interpretable (`docs/RESEARCH_PLAN.md` Stage D, decision D12).

| Comparison | What it isolates |
|---|---|
| TFT + EWC vs. MLP + EWC | Contribution of the temporal architecture |
| TFT + EWC vs. TFT fine-tuning | Contribution of EWC on the TFT |
| TFT + EWC vs. MLP replay / LwF | Position against other continual-learning families |
| TFT + EWC vs. joint training (MLP and TFT) | Remaining gap to the upper bound |

**Model** (`src/models/tft_continual.py`): TFT components (Lim et al., 2021) adapted to sequence-to-one estimation. The input is a causal window of the battery's most recent cycles (features and missing indicators only; past SOH labels are never inputs). There are no static covariates, so every method is task-agnostic. The TFT is capacity-matched to the MLP (about 12.0k vs. 13.1k parameters), so differences reflect the architecture, not its size.

**Protocol:** as notebook 06, with the same orders and seeds for every method, two-stage validation-only tuning (D11: each backbone's learning rate, then each method's own hyperparameter, λ for TFT + EWC), and paired tests with Holm correction. All outcomes are reported as measured.

Set `EXPERIMENT_PROFILE=smoke` for a fast code check; smoke results are not for reporting.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})
FIG_DIR = RESULTS_DIR / "figures"


def savefig(fig, name):
    """Save a figure for the paper/report (results/figures is regenerated, not committed)."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

import time

from src import evaluation as ev
from src import experiments as X
from src import protocols as P

PROFILE = os.environ.get("EXPERIMENT_PROFILE", "paper")
METHODS = ("finetune", "replay", "ewc", "lwf", "tft_finetune", "tft_ewc")
if PROFILE == "paper":
    cl_config = X.CLConfig(methods=METHODS, n_orders=10, order_seed=0, seeds=(0, 1, 2, 3, 4),
                           common_kwargs={"epochs": 200, "patience": 20, "lr": 1e-3, "batch_size": 64},
                           joint_backbones=("mlp", "tft"))
    tuning_config = X.TuningConfig(n_orders=3, seeds=(0,))
    WINDOW_SENSITIVITY = (4, 16)
else:
    cl_config = X.CLConfig(methods=METHODS, n_orders=2, order_seed=0, seeds=(0,),
                           common_kwargs={"epochs": 8, "patience": 3}, joint_backbones=("mlp", "tft"))
    tuning_config = X.TuningConfig(
        grids={"replay": {"buffer_per_task": [50, 200]}, "ewc": {"ewc_lambda": [10.0, 1_000.0]},
               "lwf": {"alpha": [0.1, 1.0]}, "tft_ewc": {"ewc_lambda": [10.0, 1_000.0]}},
        backbone_grid={"lr": [1e-3, 1e-2]}, n_orders=1, seeds=(0,))
    WINDOW_SENSITIVITY = (4,)

METHOD_COLORS = {"finetune": "#2a78d6", "replay": "#eb6834", "ewc": "#1baf7a", "lwf": "#eda100",
                 "tft_finetune": "#e87ba4", "tft_ewc": "#008300"}
METHOD_LABELS = {"finetune": "MLP fine-tuning", "replay": "MLP replay", "ewc": "MLP + EWC", "lwf": "MLP LwF",
                 "tft_finetune": "TFT fine-tuning", "tft_ewc": "TFT + EWC (proposed)",
                 "joint": "MLP joint", "tft_joint": "TFT joint"}
MARKERS = {m: ("s" if m.startswith("tft") else "o") for m in METHOD_COLORS}  # shape encodes the backbone
print("profile:", PROFILE)
print(cl_config)
print(tuning_config)

## 2. Load data

In [ ]:
dp.check_data_files(DATA_ROOT, required=["ml_training_dataset.csv"])
data = dp.read_dataset(DATA_ROOT / "ml_training_dataset.csv")
data[dp.GROUP_COL] = data[dp.GROUP_COL].astype(str)
print("rows:", len(data), "| groups:", dp.check_battery_groups(data[dp.GROUP_COL]))

## 3. Validation-only hyperparameter tuning (D11)

Two stages, both on validation periods of orders disjoint from the evaluation orders (test periods are not read):

1. **Backbone learning rate**, selected separately for the MLP and the TFT from the same grid (equal budget) with plain fine-tuning, by validation plasticity (how well each new task is learned). The TFT and the MLP need different learning rates; a shared rate would handicap one of them.
2. **Each method's own hyperparameter** (replay buffer, EWC λ, LwF α, TFT + EWC λ), by final average validation error, using the stage-1 learning rates.

A selection at a grid edge must be resolved (extend the grid) before reporting.

In [ ]:
start = time.perf_counter()
tuning = X.tune_hyperparameters(data, cl_config, tuning_config)
print(f"tuning: {len(tuning['grid_runs'])} streams in {time.perf_counter() - start:.0f} s")
print("stage 1 (backbone):", tuning["backbone_selected"])
print("stage 2 (method):", tuning["selected"])
edge = {m: [k for k, v in f.items() if v] for m, f in tuning["at_grid_edge"].items()}
edge = {m: k for m, k in edge.items() if k}
if edge:
    print("WARNING: selected value at the edge of its grid for", edge, "- extend the grid and re-run before reporting.")

panels = [(f"{b.upper()} backbone", g, "lr", "#5f5e5a", "s" if b == "tft" else "o")
          for b in tuning["backbone_selected"]
          for g in [tuning["grid_summary"][tuning["grid_summary"]["stage"] == f"backbone:{b}"]]]
panels += [(METHOD_LABELS[m], g, next(iter(tuning_config.grids[m])), METHOD_COLORS[m], MARKERS.get(m, "o"))
           for m in cl_config.methods if m in tuning["selected"]
           for g in [tuning["grid_summary"][(tuning["grid_summary"]["stage"] == "method") & (tuning["grid_summary"]["method"] == m)]]]
fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.2), squeeze=False)
for ax, (title, g, param, color, marker) in zip(axes[0], panels):
    ycol = "val_plasticity" if param == "lr" else "val_final_avg_error"
    ax.plot(g[f"param_{param}"].astype(float), g[ycol], marker=marker, markersize=6, color=color)
    sel = g[g["selected"]]
    ax.plot(sel[f"param_{param}"].astype(float), sel[ycol], marker="o", markersize=11,
            markerfacecolor="none", markeredgecolor="#1a1a19", markeredgewidth=1.5, linestyle="none")
    ax.set_xscale("log")
    ax.set_xlabel(param)
    ax.set_ylabel("val plasticity (SOH pp)" if param == "lr" else "val final avg error (SOH pp)", fontsize=8)
    ax.set_title(title, loc="left", fontsize=10)
fig.suptitle("Validation-only tuning: stage 1 backbone learning rate, stage 2 method hyperparameter (circled = selected)")
plt.tight_layout()
plt.show()
cl_config = tuning["tuned_cl_config"]

## 4. Evaluation grid

All methods on the same evaluation orders and seeds, plus joint training for both backbones.

In [ ]:
start = time.perf_counter()
cl = X.run_cl_experiment(data, cl_config)
runs, joint = cl["runs"], cl["joint"]
print(f"{len(runs)} streams in {time.perf_counter() - start:.0f} s")
summary = ev.summarise_runs(runs, list(X.CL_METRIC_COLS), ["method"]).set_index("method").reindex(list(METHODS))
joint_ci = {}
for name, g in joint.groupby("method"):
    joint_ci[name] = ev.bootstrap_ci(g.groupby("seed")["error"].mean().to_numpy())
display(summary.round(3))
for name, (m, lo, hi) in joint_ci.items():
    print(f"{METHOD_LABELS[name]} final average error: {m:.3f} [{lo:.3f}, {hi:.3f}]")

### Reuse categories after each step (E10, D10)

Classes from predicted vs. measured SOH on each task's test period, one sample per RPT interval. `reuse_final_macro_f1` / `reuse_final_accuracy` are averaged over tasks after the last step; `reuse_forgetting` and `reuse_backward_transfer` use 1 − macro-F1, so positive values mean category forgetting. The pooled confusion matrix (rows measured, columns predicted; C0–C3) sums the last step over all runs.

In [ ]:
reuse_summary = ev.summarise_runs(runs, list(X.REUSE_METRIC_COLS), ["method"]).set_index("method")
display(reuse_summary.round(3))
for m, conf in cl["reuse_confusion"].items():
    print(METHOD_LABELS.get(m, m))
    display(pd.DataFrame(conf, index=[f"measured {c.split()[0]}" for c in ev.REUSE_CLASS_NAMES],
                         columns=[f"pred {c.split()[0]}" for c in ev.REUSE_CLASS_NAMES]))

## 5. Ablations: paired comparisons against TFT + EWC

Differences are **other method − TFT + EWC** on matched (order, seed) runs, so **positive values mean TFT + EWC has lower error / less forgetting**. The Holm family is all comparisons against TFT + EWC for one metric.

In [ ]:
ABLATION = {"ewc": "architecture (MLP + EWC)", "tft_finetune": "EWC on the TFT (TFT fine-tuning)",
            "replay": "vs. replay", "lwf": "vs. LwF", "finetune": "vs. MLP fine-tuning"}
comparisons = X.compare_methods(runs, baseline="tft_ewc")
comparisons.insert(1, "ablation", comparisons["method"].map(ABLATION))
key_metrics = ["final_avg_error", "forgetting", "plasticity"]
display(comparisons[comparisons["metric"].isin(key_metrics)][
    ["metric", "ablation", "n", "n_nonzero", "mean_diff", "ci_low", "ci_high", "p_value", "p_holm", "rank_biserial"]].round(4))

gap = {name: summary.loc["tft_ewc", "final_avg_error_mean"] - m for name, (m, _, _) in joint_ci.items()}
print("Gap to joint training (TFT + EWC final avg error minus joint):", {METHOD_LABELS[k]: round(float(v), 3) for k, v in gap.items()})

## 6. Adaptation–retention trade-off

Circles are MLP methods, squares TFT methods; error bars are 95 % CIs over orders × seeds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
for m in METHODS:
    r = summary.loc[m]
    ax.errorbar(r["plasticity_mean"], r["forgetting_mean"],
                xerr=[[r["plasticity_mean"] - r["plasticity_ci_low"]], [r["plasticity_ci_high"] - r["plasticity_mean"]]],
                yerr=[[r["forgetting_mean"] - r["forgetting_ci_low"]], [r["forgetting_ci_high"] - r["forgetting_mean"]]],
                fmt=MARKERS[m], markersize=8, color=METHOD_COLORS[m], ecolor=METHOD_COLORS[m], elinewidth=1.3, capsize=3,
                label=METHOD_LABELS[m])
ax.axhline(0, color="#8a8981", linewidth=1)
ax.set_xlabel("Plasticity: error on newest task (SOH pp)")
ax.set_ylabel("Forgetting (SOH pp)")
ax.set_title("Adaptation vs. retention", loc="left")
ax.legend(frameon=False, fontsize=8, loc="best")

ax = axes[1]
xs = np.arange(len(METHODS))
means = summary["final_avg_error_mean"].to_numpy()
err = [means - summary["final_avg_error_ci_low"].to_numpy(), summary["final_avg_error_ci_high"].to_numpy() - means]
ax.bar(xs, means, color=[METHOD_COLORS[m] for m in METHODS], width=0.65)
ax.errorbar(xs, means, yerr=err, fmt="none", ecolor="#3d3d3a", capsize=3, linewidth=1)
for name, style in [("joint", "--"), ("tft_joint", ":")]:
    if name in joint_ci:
        ax.axhline(joint_ci[name][0], color="#5f5e5a", linestyle=style, linewidth=1.5, label=METHOD_LABELS[name])
ax.set_xticks(xs, [METHOD_LABELS[m].replace(" (proposed)", "*") for m in METHODS], rotation=25, ha="right", fontsize=8)
ax.set_ylabel("Final average error (SOH pp)")
ax.set_title("Error on all groups after the last task (* proposed)", loc="left")
ax.legend(frameon=False, fontsize=8, loc="upper right")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

## 7. Interpretation of TFT + EWC

One stream (identity order, first seed, tuned λ) is re-run and the trained model is inspected on every task's test period: mean variable-selection weights at the estimated cycle, and mean attention of the estimated cycle over the preceding cycles in its window. These describe what the model uses; they are not causal explanations of degradation.

In [ ]:
order0 = cl["orders"][0]
tasks0 = P.make_task_stream(data, order0, cl_config.train_frac, cl_config.val_frac)
kwargs = X.method_settings(cl_config, "tft_ewc")
fitted = X.run_stream("tft_ewc", tasks0, cl_config.seeds[0], return_strategy=True, **kwargs)["strategy"]
all_test = pd.concat([t.test for t in tasks0], ignore_index=True)
interp = fitted.interpret(all_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.3, 1]})
vw = interp["variable_weights"].sort_values()
axes[0].barh(vw.index, vw.values, color="#008300", height=0.7)
axes[0].set_xlabel("Mean variable-selection weight")
axes[0].set_title("Variables used at the estimated cycle", loc="left")
axes[0].tick_params(axis="y", labelsize=8)
axes[0].grid(axis="y", visible=False)
lag = interp["attention_by_lag"]
axes[1].bar(lag.index, lag.values, color="#008300", width=0.7)
axes[1].set_xlabel("Cycle lag (0 = estimated cycle)")
axes[1].set_ylabel("Mean attention")
axes[1].set_title("Attention over the cycle window", loc="left")
axes[1].grid(axis="x", visible=False)
plt.tight_layout()
plt.show()
display(pd.DataFrame({"weight": interp["variable_weights"]}).round(4))

## 8. Sensitivity to the window length

The window length (8 cycles) is fixed a priori. TFT + EWC is re-run with other window lengths on the same evaluation orders and the first seed, with the tuned λ, and compared with window 8 on the same runs.

In [ ]:
base = runs[(runs["method"] == "tft_ewc") & (runs["seed"] == cl_config.seeds[0])].assign(window=8)
sens = [base]
for w in WINDOW_SENSITIVITY:
    cfg_w = X.CLConfig(**{**cl.get("config", {}), "methods": ("tft_ewc",), "seeds": (cl_config.seeds[0],),
                          "joint_backbones": (), "method_kwargs": {"tft_ewc": {**cl_config.method_kwargs.get("tft_ewc", {}), "window": w}}})
    sens.append(X.run_cl_experiment(data, cfg_w)["runs"].assign(window=w))
window_sensitivity = ev.summarise_runs(pd.concat(sens, ignore_index=True), ["final_avg_error", "plasticity", "forgetting"], ["window"])
window_sensitivity.round(3)

## 9. Save results

In [ ]:
matrices_out = [{"method": m, "order_id": o, "seed": sd, "R": R.round(6).to_dict(orient="split")}
                for (m, o, sd), R in cl["matrices"].items()]
out = X.write_results(RESULTS_DIR / "tft_ewc_metrics.json", {
    "profile": PROFILE, "config": cl["config"], "orders": cl["orders"],
    "tuning": {k: tuning[k] for k in ("config", "orders", "selected", "at_grid_edge", "grid_summary", "grid_runs")},
    "runs": runs, "summary": summary.reset_index(), "joint": joint, "comparisons_vs_tft_ewc": comparisons,
    "reuse_summary": reuse_summary.reset_index(), "reuse_confusion": {m: c.tolist() for m, c in cl["reuse_confusion"].items()},
    "interpretation": {k: v.to_dict() for k, v in interp.items()}, "window_sensitivity": window_sensitivity,
    "matrices": matrices_out,
}, REPO_ROOT)
print("wrote", out.relative_to(REPO_ROOT), f"({out.stat().st_size / 1e3:.0f} kB)")
if PROFILE != "paper":
    print("NOTE: smoke profile — these numbers are for code checks only.")

## 10. Caveats

* **Six groups.** CIs cover variability over orders and seeds, not other battery populations.
* **Small data per task.** Each task has on the order of a hundred training cycles; a sequence model can overfit where a per-cycle model does not. Capacity matching limits but does not remove this, so the ablation against MLP + EWC is essential to any claim about the architecture.
* **Interpretation** (variable weights, attention) describes the fitted model on this data, not physical causes of degradation.
* **Fixed settings.** Epochs, patience, batch size, window length and model width are fixed a priori (the learning rate is tuned per backbone); the window sensitivity above shows how much the TFT result depends on one of them.